# Figure 2 FC-IS reproducibility supplement | Supplementary Fig. S2a

This notebook reproduces the FC–IS analysis across MLP, CNN and Transformer models, multiple datasets and 15 independently initialized seeds. The architecture-level comparisons are reported in Extended Data Fig. 2a.

In [ ]:
from collections import Counter
from dataclasses import asdict
import hashlib
import json
from pathlib import Path
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


def locate_supplementary_code(start: Path) -> Path:
    start = start.resolve()
    candidates = []
    for base in (start, *start.parents):
        candidates.extend([
            base,
            base / 'Supplementary_fig_code',
            base / 'FC-IS_code' / 'Supplementary_fig_code',
        ])
    for candidate in candidates:
        if (candidate / 'utils' / 'fig2_fc_is.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not locate FC-IS_code/Supplementary_fig_code. '
        'Start Jupyter from the project root or the supplementary-code folder.'
    )


CODE_DIR = locate_supplementary_code(Path.cwd())
FC_IS_CODE_ROOT = CODE_DIR.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from utils.models import MLP, SimpleCNN, TransformerLM, load_state_dict_file
from utils.fig2_fc_is import (
    SuppFig3Config,
    analyze_cnn_run,
    analyze_mlp_run,
    analyze_transformer_run,
    assemble_results,
    initialize_cnn_plateau,
    initialize_transformer_xavier,
    load_run_records,
    plot_fig2_fc_is_supp,
    safe_slug,
    save_results,
    save_run_records,
    set_seed,
    train_cnn_model,
    train_mlp_model,
    train_transformer_model,
    validate_results,
)

print('Supplementary code:', CODE_DIR)
print('FC-IS code root:', FC_IS_CODE_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. Configuration and relative paths

All data paths are derived from the existing `FC-IS_code` layout. No workstation or server absolute path is embedded in the notebook. The default seed set contains 15 independent runs for every architecture–dataset combination.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG = SuppFig3Config(
    seeds=tuple(range(15)),
    analysis_seed=42,
    minimum_publication_seeds=15,
    mlp_eval_samples=1000,
    cnn_eval_samples=500,
    cnn_units_per_layer=256,
    transformer_eval_samples=64,
    transformer_jacobian_samples=32,
    transformer_layers=(1,),
    transformer_chunk_size=2,
    association='pearson',
)

MLP_DATA_ROOT = FC_IS_CODE_ROOT / 'MLP' / 'data'
CNN_DATA_ROOT = FC_IS_CODE_ROOT / 'CNN' / 'data'
TRANSFORMER_DATA_ROOT = FC_IS_CODE_ROOT / 'Transformer' / 'data'
CHECKPOINT_ROOT = CODE_DIR / 'checkpoints' / 'fig2_FC_IS_reproducibility_supp'
RUN_RESULT_ROOT = CODE_DIR / 'results' / 'fig2_FC_IS_reproducibility_supp' / 'fig2_FC_IS_reproducibility_supp' / 'runs'
RESULT_ROOT = CODE_DIR / 'results' / 'fig2_FC_IS_reproducibility_supp'
OUTPUT_ROOT = CODE_DIR / 'outputs' / 'fig2_FC_IS_reproducibility_supp'
for folder in (CHECKPOINT_ROOT, RUN_RESULT_ROOT, RESULT_ROOT, OUTPUT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

DOWNLOAD_IMAGE_DATA = False
TRAIN_MISSING_CHECKPOINTS = True
FORCE_RETRAIN = False
FORCE_REANALYZE = False
CACHE_VERSION = 'primary_layers_only_v3'
analysis_signature = hashlib.sha256(
    json.dumps({'version': CACHE_VERSION, 'config': asdict(CONFIG)}, sort_keys=True).encode('utf-8')
).hexdigest()[:12]

REFERENCE_DATASETS = {
    'MLP': 'MNIST',
    'CNN': 'MNIST',
    'Transformer': 'WikiText-2',
}
DATASET_ORDER = {
    'MLP': ('MNIST', 'FashionMNIST', 'CIFAR10'),
    'CNN': ('MNIST', 'FashionMNIST', 'CIFAR10'),
    'Transformer': ('WikiText-2', 'Penn Treebank'),
}
PRIMARY_LAYERS = {'MLP': 'H2', 'CNN': 'C2', 'Transformer': 'L2'}

CONFIG

## 2. Image datasets

MLP and CNN are tested on MNIST, FashionMNIST and CIFAR10. Normalization follows the corresponding reference loaders. MNIST and FashionMNIST are resized to 32 × 32 only for CNN, matching the existing CNN input geometry; CIFAR10 remains 32 × 32 with three input channels. Set `DOWNLOAD_IMAGE_DATA=True` only if the server can access the dataset hosts. Otherwise place the torchvision datasets under the existing `MLP/data` and `CNN/data` roots.

In [ ]:
IMAGE_SPECS = {
    'MNIST': {
        'dataset_class': datasets.MNIST,
        'mean': (0.1307,),
        'std': (0.3081,),
        'input_size': 784,
        'in_channels': 1,
    },
    'FashionMNIST': {
        'dataset_class': datasets.FashionMNIST,
        'mean': (0.2860,),
        'std': (0.3530,),
        'input_size': 784,
        'in_channels': 1,
    },
    'CIFAR10': {
        'dataset_class': datasets.CIFAR10,
        'mean': (0.4914, 0.4822, 0.4465),
        'std': (0.2023, 0.1994, 0.2010),
        'input_size': 3072,
        'in_channels': 3,
    },
}


def seeded_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        drop_last=False,
    )


def build_image_datasets(dataset_name, architecture):
    spec = IMAGE_SPECS[dataset_name]
    transform_steps = []
    if architecture == 'CNN' and dataset_name in {'MNIST', 'FashionMNIST'}:
        transform_steps.append(transforms.Resize((32, 32)))
    transform_steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(spec['mean'], spec['std']),
    ])
    root = MLP_DATA_ROOT if architecture == 'MLP' else CNN_DATA_ROOT
    dataset_class = spec['dataset_class']
    train_dataset = dataset_class(
        root=root, train=True, transform=transforms.Compose(transform_steps),
        download=DOWNLOAD_IMAGE_DATA,
    )
    eval_dataset = dataset_class(
        root=root, train=False, transform=transforms.Compose(transform_steps),
        download=DOWNLOAD_IMAGE_DATA,
    )
    return train_dataset, eval_dataset, spec

## 3. Transformer corpora and manual download

The reference corpus remains WikiText-2. Penn Treebank is added as an independent language-modeling dataset rather than another WikiText scale variant. No network download is performed by this notebook.

Create the following relative directory and save the three files with exactly these names:

```text
FC-IS_code/Transformer/data/penn-treebank/
├── ptb.train.txt
├── ptb.valid.txt
└── ptb.test.txt
```

Dataset documentation: https://docs.pytorch.org/text/stable/_modules/torchtext/datasets/penntreebank.html

Direct files listed by the PyTorch dataset implementation:

- https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.train.txt
- https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.valid.txt
- https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt

Keep the existing WikiText-2 layout used by the reference Transformer code:

```text
FC-IS_code/Transformer/data/wikitext-2/wiki.train.tokens
FC-IS_code/Transformer/data/wikitext-2/wiki.valid.tokens
```

In [ ]:
TRANSFORMER_CORPORA = {
    'WikiText-2': {
        'folder': TRANSFORMER_DATA_ROOT / 'wikitext-2',
        'train_file': 'wiki.train.tokens',
        'validation_file': 'wiki.valid.tokens',
    },
    'Penn Treebank': {
        'folder': TRANSFORMER_DATA_ROOT / 'penn-treebank',
        'train_file': 'ptb.train.txt',
        'validation_file': 'ptb.valid.txt',
    },
}


class WordSequenceDataset(Dataset):
    def __init__(self, text, vocab, seq_len=32):
        token_ids = [vocab.get(word, 1) for word in text.split()]
        total = (len(token_ids) // seq_len) * seq_len
        self.data = torch.tensor(token_ids[:total], dtype=torch.long).view(-1, seq_len)

    def __len__(self):
        return max(len(self.data) - 1, 0)

    def __getitem__(self, index):
        return self.data[index], self.data[index + 1]


def build_transformer_datasets(dataset_name, vocab_size=5000, seq_len=32):
    spec = TRANSFORMER_CORPORA[dataset_name]
    train_path = spec['folder'] / spec['train_file']
    validation_path = spec['folder'] / spec['validation_file']
    missing = [path for path in (train_path, validation_path) if not path.is_file()]
    if missing:
        expected = '\n'.join(str(path.relative_to(FC_IS_CODE_ROOT)) for path in missing)
        raise FileNotFoundError(
            f'Missing local files for {dataset_name}:\n{expected}\n'
            'Use the manual-download instructions in the preceding cell.'
        )
    train_text = train_path.read_text(encoding='utf-8')
    validation_text = validation_path.read_text(encoding='utf-8')
    frequent_words = [
        word for word, _ in Counter(train_text.split()).most_common(vocab_size - 2)
    ]
    vocab = {'<pad>': 0, '<unk>': 1}
    vocab.update({word: index + 2 for index, word in enumerate(frequent_words)})
    train_dataset = WordSequenceDataset(train_text, vocab, seq_len=seq_len)
    eval_dataset = WordSequenceDataset(validation_text, vocab, seq_len=seq_len)
    return train_dataset, eval_dataset, vocab

## 4. Cache helpers

Checkpoints are separated by architecture, dataset and seed. FC–IS run records are cached independently from checkpoints, so changing figure styling does not trigger retraining or repeated Transformer Jacobian estimation.

In [ ]:
def cache_paths(architecture, dataset_name, seed):
    group = Path(safe_slug(architecture)) / safe_slug(dataset_name)
    checkpoint = CHECKPOINT_ROOT / group / f'seed_{seed:02d}.pt'
    run_result = RUN_RESULT_ROOT / group / f'seed_{seed:02d}_{analysis_signature}.npz'
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    run_result.parent.mkdir(parents=True, exist_ok=True)
    return checkpoint, run_result


def load_cached_run(architecture, dataset_name, seed):
    _, run_result = cache_paths(architecture, dataset_name, seed)
    if run_result.is_file() and not FORCE_REANALYZE and not FORCE_RETRAIN:
        print(f'Loaded analysis: {architecture} | {dataset_name} | seed {seed}')
        return load_run_records(run_result)
    return None


def clear_cuda():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. MLP: 15 seeds × 3 datasets

In [ ]:
mlp_records = []
for dataset_name in DATASET_ORDER['MLP']:
    train_dataset, eval_dataset, spec = build_image_datasets(dataset_name, 'MLP')
    eval_loader = seeded_loader(eval_dataset, 128, False, CONFIG.analysis_seed)
    for seed in CONFIG.seeds:
        cached = load_cached_run('MLP', dataset_name, seed)
        if cached is not None:
            mlp_records.extend(cached)
            continue
        print(f'MLP | {dataset_name} | seed {seed}')
        set_seed(seed)
        checkpoint, run_result = cache_paths('MLP', dataset_name, seed)
        model = MLP(
            input_size=spec['input_size'], hidden_dims=(100, 100), num_classes=10,
            dropout_p=0.0, activation='relu', init_weights=True, init_method='normal',
        )
        if checkpoint.is_file() and not FORCE_RETRAIN:
            load_state_dict_file(model, str(checkpoint), map_location='cpu')
        elif TRAIN_MISSING_CHECKPOINTS:
            train_loader = seeded_loader(train_dataset, 256, True, seed)
            model = train_mlp_model(
                model, train_loader, epochs=5, learning_rate=0.05, device=DEVICE
            )
            torch.save(model.state_dict(), checkpoint)
        else:
            raise FileNotFoundError(checkpoint)
        records = analyze_mlp_run(
            model, eval_loader, seed, dataset_name, CONFIG, device=DEVICE
        )
        save_run_records(records, run_result)
        mlp_records.extend(records)
        del model
        clear_cuda()

len(mlp_records)

## 6. CNN: 15 seeds × 3 datasets

The same fixed element indices are used for every seed and dataset within a CNN layer.

In [ ]:
cnn_records = []
for dataset_name in DATASET_ORDER['CNN']:
    train_dataset, eval_dataset, spec = build_image_datasets(dataset_name, 'CNN')
    eval_loader = seeded_loader(eval_dataset, 64, False, CONFIG.analysis_seed)
    input_shape = (spec['in_channels'], 32, 32)
    for seed in CONFIG.seeds:
        cached = load_cached_run('CNN', dataset_name, seed)
        if cached is not None:
            cnn_records.extend(cached)
            continue
        print(f'CNN | {dataset_name} | seed {seed}')
        set_seed(seed)
        checkpoint, run_result = cache_paths('CNN', dataset_name, seed)
        model = initialize_cnn_plateau(
            SimpleCNN(in_channels=spec['in_channels'], activation='relu')
        )
        if checkpoint.is_file() and not FORCE_RETRAIN:
            load_state_dict_file(model, str(checkpoint), map_location='cpu')
        elif TRAIN_MISSING_CHECKPOINTS:
            train_loader = seeded_loader(train_dataset, 256, True, seed)
            model = train_cnn_model(
                model, train_loader, epochs=2, learning_rate=0.03,
                momentum=0.9, device=DEVICE,
            )
            torch.save(model.state_dict(), checkpoint)
        else:
            raise FileNotFoundError(checkpoint)
        records = analyze_cnn_run(
            model, eval_loader, seed, dataset_name, CONFIG,
            input_shape=input_shape, device=DEVICE,
        )
        save_run_records(records, run_result)
        cnn_records.extend(records)
        del model
        clear_cuda()

len(cnn_records)

## 7. Transformer: 15 seeds × 2 independent corpora

Each corpus builds its own 5,000-token vocabulary. Architecture, sequence length, optimizer and training schedule remain unchanged. Jacobian-based operational SC and incoming-profile IS follow the main Transformer analysis.

In [ ]:
transformer_records = []
for dataset_name in DATASET_ORDER['Transformer']:
    train_dataset, eval_dataset, vocab = build_transformer_datasets(
        dataset_name, vocab_size=5000, seq_len=32
    )
    eval_loader = seeded_loader(eval_dataset, 16, False, CONFIG.analysis_seed)
    for seed in CONFIG.seeds:
        cached = load_cached_run('Transformer', dataset_name, seed)
        if cached is not None:
            transformer_records.extend(cached)
            continue
        print(f'Transformer | {dataset_name} | seed {seed}')
        set_seed(seed)
        checkpoint, run_result = cache_paths('Transformer', dataset_name, seed)
        model = initialize_transformer_xavier(TransformerLM(
            vocab_size=5000, d_model=128, n_heads=4, d_ff=256,
            n_layers=2, seq_len=32, dropout=0.1,
        ))
        if checkpoint.is_file() and not FORCE_RETRAIN:
            load_state_dict_file(model, str(checkpoint), map_location='cpu')
        elif TRAIN_MISSING_CHECKPOINTS:
            train_loader = seeded_loader(train_dataset, 64, True, seed)
            model = train_transformer_model(
                model, train_loader, epochs=3, learning_rate=1e-4, device=DEVICE
            )
            torch.save(model.state_dict(), checkpoint)
        else:
            raise FileNotFoundError(checkpoint)
        records = analyze_transformer_run(
            model, eval_loader, seed, dataset_name, CONFIG, device=DEVICE
        )
        save_run_records(records, run_result)
        transformer_records.extend(records)
        del model
        clear_cuda()

len(transformer_records)

## 8. Validate, save source data and export Supplementary Fig. S2a

In [ ]:
all_records = mlp_records + cnn_records + transformer_records
results = assemble_results(
    all_records,
    CONFIG,
    metadata={
        'reference_datasets': REFERENCE_DATASETS,
        'dataset_order': {key: list(value) for key, value in DATASET_ORDER.items()},
        'primary_layers': PRIMARY_LAYERS,
        'path_policy': 'relative to the existing FC-IS_code directory',
        'transformer_corpora': {
            'WikiText-2': 'local wiki.train.tokens and wiki.valid.tokens',
            'Penn Treebank': 'local ptb.train.txt and ptb.valid.txt',
        },
    },
)
validate_results(
    results,
    minimum_seeds=CONFIG.minimum_publication_seeds,
    required_groups=DATASET_ORDER,
)
result_path = save_results(
    results, RESULT_ROOT / 'fig2_FC_IS_reproducibility_supp_source_data.npz'
)
figure, statistics = plot_fig2_fc_is_supp(
    results,
    output_prefix=OUTPUT_ROOT / 'fig2_FC_IS_reproducibility_supp',
    dataset_order=DATASET_ORDER,
    primary_layers=PRIMARY_LAYERS,
    minimum_seeds=CONFIG.minimum_publication_seeds,
    export_formats=('svg', 'pdf', 'tiff', 'png'),
    dpi=600,
)
display(figure)
print('Saved source data:', result_path)
statistics

## Reporting checklist

- Extended Data Fig. 2a summarizes FC–IS reproducibility across datasets and independently trained seeds.
- Primary layers are MLP H2, CNN C2 and Transformer L2.
- Other Extended Data Fig. 2 panels are generated by separate notebooks.